# Experiment Design

## Overview

Good experiment design eliminates confounding, maximises power for a given budget, and ensures results generalise to the target population. Design decisions made before data collection cannot be corrected in analysis.

**Design choices:**

| Design | When | Advantage |
|---|---|---|
| Completely randomised | Units exchangeable | Simple, maximum flexibility |
| Randomised complete block | Known nuisance variable | Removes block variance |
| Stratified | Subgroup representation needed | Balanced subgroups |
| Factorial | Multiple factors | Tests interactions |
| Crossover | Subjects as own control | Highest efficiency |
| Sequential | Monitor as data arrives | Stop early if clear signal |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)
print("Ecological restoration experiment designs")
print("Units: riparian sites; Outcome: species richness")

---
## Completely Randomised vs Randomised Block Design

In [ ]:
# True data-generating process: catchment is a strong nuisance variable
n_blocks    = 6    # catchments
n_per_block = 8    # sites per catchment (4 control, 4 treatment)
true_effect = 3.0  # species uplift from restoration
catchment_effects = rng.normal(0, 6, n_blocks)   # large between-catchment variation

records = []
for blk in range(n_blocks):
    for unit in range(n_per_block):
        trt = 1 if unit < n_per_block//2 else 0
        richness = (15 + catchment_effects[blk] + trt*true_effect
                    + rng.normal(0, 2))
        records.append({'catchment': blk, 'treatment': trt, 'richness': richness})
df = pd.DataFrame(records)

# CRD: ignore catchment structure
ctrl_crd = df[df.treatment==0]['richness']
trt_crd  = df[df.treatment==1]['richness']
_, p_crd = stats.ttest_ind(ctrl_crd, trt_crd)

# RBD: include catchment as blocking factor
model_rbd = ols('richness ~ C(treatment) + C(catchment)', data=df).fit()
p_rbd = model_rbd.pvalues['C(treatment)[T.1]']
print(f"True effect: {true_effect:.1f} species")
print(f"CRD (ignoring catchment): p={p_crd:.4f}")
print(f"RBD (blocking on catchment): p={p_rbd:.4f}")
print("RBD removes catchment variance -> higher power for same n")

---
## Factorial Design

In [ ]:
# 2×2 factorial: restoration type × buffer width
n_cell = 20
designs = {'planted': 0, 'seeded': 1}
widths  = {'narrow': 0, 'wide': 1}

factorial_records = []
for (dname, d), (wname, w) in [(x,y) for x in designs.items() for y in widths.items()]:
    richness = (15 + 2.5*d + 3.0*w + 1.5*d*w   # interaction: wide+planted extra benefit
                + rng.normal(0, 3, n_cell))
    for r in richness:
        factorial_records.append({'design': dname, 'width': wname,
                                   'design_code': d, 'width_code': w, 'richness': r})
df_fact = pd.DataFrame(factorial_records)
model_f = ols('richness ~ C(design) * C(width)', data=df_fact).fit()
print(anova_lm(model_f, typ=2).round(4))

# Interaction plot
fig, ax = plt.subplots(figsize=(7,4))
for wname, grp in df_fact.groupby('width'):
    means = grp.groupby('design')['richness'].mean()
    ax.plot(means.index, means.values, 'o-', lw=2, ms=8, label=f'Width: {wname}')
    for dname, val in means.items():
        ax.annotate(f'{val:.1f}', (dname, val), textcoords='offset points',
                    xytext=(5,5), fontsize=9)
ax.set_xlabel('Restoration design'); ax.set_ylabel('Mean richness')
ax.set_title('Interaction Plot: Design × Buffer Width')
ax.legend(); plt.tight_layout(); plt.show()

---
## Stratified Randomisation

In [ ]:
# Stratified randomisation: ensure balance on elevation strata
# (High elevation sites naturally lower richness -> must balance across arms)
n_sites  = 80
elevation = rng.uniform(50, 400, n_sites)
strata   = pd.cut(elevation, bins=[50,175,250,400], labels=['low','mid','high'])
df_strat = pd.DataFrame({'site_id': range(n_sites), 'elevation': elevation,
                          'stratum': strata})

# Simple randomisation (may be unbalanced within strata)
df_strat['simple_arm'] = rng.choice(['ctrl','trt'], n_sites)

# Stratified randomisation: randomise within each stratum
df_strat['strat_arm'] = 'ctrl'
for stratum, grp in df_strat.groupby('stratum', observed=True):
    idx = grp.index.tolist()
    rng.shuffle(idx)
    half = len(idx)//2
    df_strat.loc[idx[:half], 'strat_arm'] = 'trt'

print("Balance check — sites per stratum and arm:")
print("\nSimple randomisation:")
print(pd.crosstab(df_strat['stratum'], df_strat['simple_arm']))
print("\nStratified randomisation:")
print(pd.crosstab(df_strat['stratum'], df_strat['strat_arm']))

In [ ]:
# Sequential design: group sequential with O'Brien-Fleming boundaries
# Simulate interim analyses at 25%, 50%, 75%, 100% of planned n
import numpy as np

n_total  = 200    # per arm
n_looks  = 4
look_ns  = [50, 100, 150, 200]
true_eff = 2.5

# O'Brien-Fleming spending function boundaries (approximate)
# Nominal α at each interim to preserve overall α=0.05
alpha_spend = [0.0005, 0.014, 0.045, 0.05]   # cumulative spending (approx OF)

# Simulate trial
ctrl_all = rng.normal(15.0, 5.0, n_total)
trt_all  = rng.normal(15.0 + true_eff, 5.0, n_total)

print("Sequential analysis (O'Brien-Fleming boundaries):")
print(f"{'Look':>6} {'n/arm':>7} {'p-value':>10} {'α spend':>10} {'Decision':>12}")
stopped = False
for look_n, a_sp in zip(look_ns, alpha_spend):
    _, p = stats.ttest_ind(trt_all[:look_n], ctrl_all[:look_n])
    decision = 'STOP (sig)' if p < a_sp else ('STOP (futile)' if p > 0.9 else 'Continue')
    print(f"{look_ns.index(look_n)+1:6d} {look_n:7d} {p:10.4f} {a_sp:10.4f} {decision:>12}")
    if 'STOP' in decision:
        stopped = True; break
if not stopped:
    print("Trial completed without early stopping")

---

## Common Pitfalls

**1. Randomising without stratification when strong prognostic factors exist**  
In small experiments, chance imbalance on a strong prognostic variable (e.g. elevation, soil type, catchment) inflates variance and can produce confounded results even with randomisation. Stratify randomisation on known strong predictors to guarantee balance across arms.

**2. Analysing a blocked design as if it were completely randomised**  
If sites were assigned within catchment blocks but analysed with a simple t-test ignoring blocks, the between-catchment variance inflates the error term, reducing power and potentially biasing the p-value. Always analyse the design you randomised — include block effects in the model.

**3. Testing main effects from a factorial design without first examining the interaction**  
If an interaction is present (e.g. wide buffers only help with planted restoration), main effect estimates are misleading averages across levels where the effect is heterogeneous. Always plot and test the interaction term first; if significant, interpret main effects conditionally.

**4. Running interim analyses without a pre-specified stopping rule**  
Looking at accumulating data and stopping when p < 0.05 inflates the Type I error rate substantially — with four looks at α=0.05 each, the overall error rate approaches 14%. Pre-specify the number of interim analyses, timing, and alpha-spending function (e.g. O'Brien-Fleming) before data collection.

**5. Failing to distinguish the experimental unit from the observation unit**  
If restoration is applied to whole catchments but individual sites within catchments are analysed as independent observations, the standard error is underestimated and p-values are anti-conservative. The experimental unit (the unit receiving the treatment) determines the effective sample size for inference.
---
*python_methods_library - Samantha McGarrigle*